In [1]:
from pathlib import Path
import pandas as pd

src = Path("Merged_Dataset") / "icu_patients(re-assign).csv"
df = pd.read_csv(src, low_memory=False)

df = df.loc[:, ["subject_id", "assigned_intime", "assigned_outtime"]].copy()

for col in ["assigned_intime", "assigned_outtime"]:
    df[col] = pd.to_datetime(df[col], errors="coerce").dt.strftime("%Y-%m-%d")


In [2]:
# Generate date dimension table, compute N (inpatients on that day), inflow (admissions), outflow (discharges)
df2 = df.copy()
df2['assigned_intime'] = pd.to_datetime(df2['assigned_intime'], errors='coerce')
df2[col] = pd.to_datetime(df2[col], errors='coerce')  # col = 'assigned_outtime'

start = df2['assigned_intime'].min().normalize()
end = pd.to_datetime(max(df2['assigned_intime'].max(), df2[col].max())).normalize()

date_index = pd.date_range(start, end, freq='D')
date_df = pd.DataFrame({'date': date_index})

# admissions/discharges on each day
inflow = df2.groupby(df2['assigned_intime'].dt.normalize()).size().reindex(date_index, fill_value=0)
outflow = df2.groupby(df2[col].dt.normalize()).size().reindex(date_index, fill_value=0)

# inpatients N on each day: admitted before the day and discharged after the day (or not discharged yet)
N = []
for d in date_index:
    mask = (df2['assigned_intime'] < d) & (df2[col].isna() | (df2[col] >= d))
    N.append(int(mask.sum()))

date_df['N'] = N
date_df['inflow'] = inflow.values
date_df['outflow'] = outflow.values

# occupancy = N + inflow - outflow
date_df['occupancy'] = date_df['N'] + date_df['inflow'] - date_df['outflow']

# set date as index (optional)
date_df = date_df.set_index('date')

# keep only rows between 2020-05-19 and 2023-12-24 (inclusive)
start_keep = "2020-05-19"
end_keep = "2023-12-31"

date_df = date_df.loc[start_keep:end_keep].copy()
date_index = date_df.index

N = date_df['N'].tolist()
inflow = date_df['inflow']
outflow = date_df['outflow']


# quick check
date_df.head(10)

,N,inflow,outflow,occupancy
date,,,,
2020-05-19,66,14,15,65
2020-05-20,65,15,16,64
2020-05-21,64,29,10,83
2020-05-22,83,23,32,74
2020-05-23,74,18,20,72
2020-05-24,72,17,21,68
2020-05-25,68,17,12,73
2020-05-26,73,19,18,74
2020-05-27,74,18,14,78


In [3]:
# check whether occupancy equals next day's N
next_N = date_df['N'].shift(-1)
eq = date_df['occupancy'] == next_N


print("Days checked (excluding last day):", len(date_df) - 1)
print("Matches:", eq.iloc[:-1].sum())
print("Mismatches:", (~eq.iloc[:-1]).sum())
print("All match (excluding last day):", bool(eq.iloc[:-1].all()))

Days checked (excluding last day): 1321
Matches: 1321
Mismatches: 0
All match (excluding last day): True


In [4]:
# Compute per-day subject_id lists (admissions, discharges, inpatients) and save only the main date_df CSV
# Note: CSV will stringify Python lists. If you need list types preserved, save parquet/json separately.
dest_dir = Path("Predictions")
dest_dir.mkdir(parents=True, exist_ok=True)

# Ensure df2 datetime columns exist
df2 = df2.copy()
df2['assigned_intime'] = pd.to_datetime(df2['assigned_intime'], errors='coerce')
df2['assigned_outtime'] = pd.to_datetime(df2['assigned_outtime'], errors='coerce')

# helper date-only columns
df2['intime_date'] = df2['assigned_intime'].dt.normalize()
df2['outtime_date'] = df2['assigned_outtime'].dt.normalize()

# maps date -> list(subject_id)
admit_map = df2.dropna(subset=['intime_date']).groupby('intime_date')['subject_id'].apply(list).to_dict()
discharge_map = df2.dropna(subset=['outtime_date']).groupby('outtime_date')['subject_id'].apply(list).to_dict()

admitted_ids = []
discharged_ids = []
inpatient_ids = []

for d in date_index:
    admitted = admit_map.get(d, [])
    discharged = discharge_map.get(d, [])
    admitted_ids.append(admitted)
    discharged_ids.append(discharged)
    # inpatients present on day d: admitted before the day (intime < d) and not discharged before the day
    mask = (df2['assigned_intime'] < d) & (df2['assigned_outtime'].isna() | (df2['assigned_outtime'] >= d))
    inpatients = df2.loc[mask, 'subject_id'].tolist()
    inpatient_ids.append(inpatients)

# assign to date_df (index is date_index)
date_df = date_df.copy()
date_df['admitted_ids'] = admitted_ids
date_df['discharged_ids'] = discharged_ids
date_df['inpatient_ids'] = inpatient_ids

# validation counts
date_df['len_admitted_ids'] = date_df['admitted_ids'].apply(len)
date_df['len_discharged_ids'] = date_df['discharged_ids'].apply(len)
date_df['len_inpatient_ids'] = date_df['inpatient_ids'].apply(len)

print('Summary checks:')
print('inflow equals len_admitted_ids ? ', (date_df['inflow'] == date_df['len_admitted_ids']).all())
print('outflow equals len_discharged_ids ? ', (date_df['outflow'] == date_df['len_discharged_ids']).all())
print('N equals len_inpatient_ids ? ', (date_df['N'] == date_df['len_inpatient_ids']).all())

# Save only the main date_df CSV as requested (lists will be stringified in CSV)
start_str = date_df.index.min().strftime("%Y-%m-%d")
end_str = date_df.index.max().strftime("%Y-%m-%d")
date_df_csv = dest_dir / f"occupancy_{start_str}_to_{end_str}.csv"
date_df.to_csv(date_df_csv)
print(f"Saved date_df CSV to: {date_df_csv}")


Summary checks:
inflow equals len_admitted_ids ?  True
outflow equals len_discharged_ids ?  True
N equals len_inpatient_ids ?  True
Saved date_df CSV to: Predictions\occupancy_2020-05-19_to_2023-12-31.csv


In [5]:
src2 = Path("Predictions") / "Results_with_assigned_chartdates(re-assign).csv"
df_results = pd.read_csv(src2, low_memory=False)

# keep needed columns (remove duplicate) and parse datetimes
required_cols = ["assigned_chartdate", "subject_id", "assigned_intime", "assigned_outtime", "actual_los", "predicted_los"]
df_results = df_results[required_cols].copy()

df_results['assigned_chartdate'] = pd.to_datetime(df_results['assigned_chartdate'], errors='coerce').dt.normalize()
# keep only the date (no time) for intime/outtime by normalizing to midnight
df_results['assigned_intime'] = pd.to_datetime(df_results['assigned_intime'], errors='coerce').dt.normalize()
df_results['assigned_outtime'] = pd.to_datetime(df_results['assigned_outtime'], errors='coerce').dt.normalize()

# set assigned_chartdate as datetime index and sort, then slice by date range
df_results = df_results.set_index('assigned_chartdate').sort_index()
df_los = df_results.loc[start_keep:end_keep].copy()

#save df_los to CSV
dest_los_csv = dest_dir / f"df_los_{start_str}_to_{end_str}.csv"
df_los.to_csv(dest_los_csv)

print(df_los.shape)
df_los.head(10)

(241282, 5)


,subject_id,assigned_intime,assigned_outtime,actual_los,predicted_los
assigned_chartdate,,,,,
2020-05-19,12230431,2020-05-18,2020-05-20,1,1.001645
2020-05-19,12230431,2020-05-18,2020-05-20,1,1.001645
2020-05-19,12238622,2020-05-18,2020-05-20,1,1.001645
2020-05-19,12238622,2020-05-18,2020-05-20,1,1.001645
2020-05-19,14501923,2020-05-18,2020-05-19,1,4.104186
2020-05-19,14501923,2020-05-18,2020-05-19,1,4.438365
2020-05-19,15523187,2020-05-18,2020-05-19,1,1.896511
2020-05-19,15523187,2020-05-18,2020-05-19,1,1.001645
2020-05-19,10501162,2020-05-18,2020-05-24,5,3.422209


In [6]:
# Output the number of unique subject_id in df_los
try:
    _df = df_los
except NameError:
    raise NameError("Variable df_los not found. Please run the cell that creates df_los first.")

# If subject_id is in the index or not present in columns, reset index to ensure accessibility
if 'subject_id' not in _df.columns:
    _df = _df.reset_index()

if 'subject_id' not in _df.columns:
    raise KeyError("Column 'subject_id' not found in df_los. Please check the structure of df_los.")

unique_subject_count = int(_df['subject_id'].nunique())
print("Number of unique subject_id in df_los:", unique_subject_count)

Number of unique subject_id in df_los: 8587


In [7]:
# Check whether df_los assigned_chartdate (as index) is continuous and list missing dates
if 'df_los' not in globals():
    raise NameError("Variable df_los not found. Please run the cell that creates df_los first.")

# Ensure the index is a DatetimeIndex (if not, try to use the assigned_chartdate column)
if not isinstance(df_los.index, pd.DatetimeIndex):
    if 'assigned_chartdate' in df_los.columns:
        present_idx = pd.to_datetime(df_los['assigned_chartdate']).dt.normalize()
    else:
        raise KeyError("df_los index is not a DatetimeIndex and column 'assigned_chartdate' was not found.")
else:
    present_idx = pd.to_datetime(df_los.index).normalize()

present_unique = pd.DatetimeIndex(sorted(pd.Series(present_idx).dropna().unique()))
if present_unique.empty:
    print("No valid date index found in df_los.")
else:
    expected = pd.date_range(present_unique.min(), present_unique.max(), freq='D')
    missing = expected.difference(present_unique)

    print(f"Date range: {present_unique.min().date()} to {present_unique.max().date()}")
    print(f"Expected days (inclusive): {len(expected)}")
    print(f"Distinct chartdate days actually present: {len(present_unique)}")
    print(f"Missing days: {len(missing)}")

    if len(missing) == 0:
        print("assigned_chartdate is continuous in this range (no missing dates).")
    else:
        print("\nExamples of missing dates (up to first 20):")
        for d in list(missing[:20]):
            print(d.date())

        # Merge consecutive missing dates into ranges for easier viewing
        miss = pd.Series(missing)
        if not miss.empty:
            diffs = miss.diff().dt.days.fillna(1).astype(int)
            ranges = []
            start = miss.iloc[0]
            prev = start
            for cur, delta in zip(miss.iloc[1:], diffs.iloc[1:]):
                if delta == 1:
                    prev = cur
                else:
                    ranges.append((start.date(), prev.date()))
                    start = cur
                    prev = cur
            ranges.append((start.date(), prev.date()))
            print("\nMissing ranges (start -> end):")
            for r in ranges[:50]:
                if r[0] == r[1]:
                    print(r[0])
                else:
                    print(f"{r[0]} -> {r[1]}")

Date range: 2020-05-19 to 2023-12-31
Expected days (inclusive): 1322
Distinct chartdate days actually present: 1322
Missing days: 0
assigned_chartdate is continuous in this range (no missing dates).


In [8]:

sel = df_results[df_results['actual_los'] == 1].copy()
sel['assigned_chartdate'] = sel.index

sel['out_minus_chart_days'] = (sel['assigned_outtime'] - sel['assigned_chartdate']).dt.days
sel['chart_minus_intime_days'] = (sel['assigned_chartdate'] - sel['assigned_intime']).dt.days
sel['out_minus_intime_days'] = (sel['assigned_outtime'] - sel['assigned_intime']).dt.days

def stats_days(series):
    s = series.dropna()
    if s.empty:
        return {'count': 0, 'min_days': None, 'max_days': None, 'mean_days': None, 'median_days': None, 'std_days': None}
    return {
        'count': int(s.count()),
        'min_days': int(s.min()),
        'max_days': int(s.max()),
        'mean_days': float(s.mean()),
        'median_days': float(s.median()),
        'std_days': float(s.std())
    }

out_chart_stats = stats_days(sel['out_minus_chart_days'])
chart_intime_stats = stats_days(sel['chart_minus_intime_days'])
out_intime_stats = stats_days(sel['out_minus_intime_days'])

print("Records with actual_los == 1 :", len(sel))
print("\nDistribution of out_minus_chart_days:")
print(sel['out_minus_chart_days'].value_counts().sort_index().head())

Records with actual_los == 1 : 45464

Distribution of out_minus_chart_days:
out_minus_chart_days
0    22980
1    22484
Name: count, dtype: int64


In [9]:
# Deduplicate: ensure each (assigned_chartdate, subject_id) has only one record.
# For actual_los and predicted_los take the mean; for other columns keep the first occurrence.
df_los = df_los.copy()

# If assigned_chartdate is the index, reset it to a column
if df_los.index.name == 'assigned_chartdate' or ('assigned_chartdate' not in df_los.columns and df_los.index.name is not None):
    df_los = df_los.reset_index()

if 'assigned_chartdate' not in df_los.columns:
    raise KeyError('Column "assigned_chartdate" not found, cannot deduplicate. Please check df_los structure.')

# Normalize dates and convert numerics so aggregation works correctly
df_los['assigned_chartdate'] = pd.to_datetime(df_los['assigned_chartdate'], errors='coerce').dt.normalize()
if 'actual_los' in df_los.columns:
    df_los['actual_los'] = pd.to_numeric(df_los['actual_los'], errors='coerce')
if 'predicted_los' in df_los.columns:
    df_los['predicted_los'] = pd.to_numeric(df_los['predicted_los'], errors='coerce')

# Build aggregation rules: mean for actual_los/predicted_los, first for other columns
agg_rules = {}
for c in df_los.columns:
    if c in ('assigned_chartdate', 'subject_id'):
        continue
    if c in ('actual_los', 'predicted_los'):
        agg_rules[c] = 'mean'
    else:
        agg_rules[c] = 'first'

before_rows = df_los.shape[0]
# groupby deduplicate
deduped = df_los.groupby(['assigned_chartdate', 'subject_id'], as_index=False).agg(agg_rules)

after_rows = deduped.shape[0]
removed = before_rows - after_rows

# Sort by date and set assigned_chartdate as index (consistent with earlier style)
deduped = deduped.sort_values('assigned_chartdate').set_index('assigned_chartdate')

# Assign back to df_los
df_los = deduped.copy()

print(f'Original rows: {before_rows}, deduplicated rows: {after_rows}, duplicates removed: {removed}')
display(df_los.head(10))

Original rows: 241282, deduplicated rows: 43285, duplicates removed: 197997


,subject_id,assigned_intime,assigned_outtime,actual_los,predicted_los
assigned_chartdate,,,,,
2020-05-19,10501162,2020-05-18,2020-05-24,5.0,6.860010
2020-05-19,19322323,2020-05-18,2020-05-22,3.0,2.371394
2020-05-19,15523187,2020-05-18,2020-05-19,1.0,1.449078
2020-05-19,14501923,2020-05-18,2020-05-19,1.0,4.271276
2020-05-19,14361391,2020-05-18,2020-05-19,1.0,1.001645
2020-05-19,18201772,2020-05-18,2020-05-21,2.0,1.001645
2020-05-19,12381582,2020-05-18,2020-05-19,1.0,1.999544
2020-05-19,12238622,2020-05-18,2020-05-20,1.0,1.001645
2020-05-19,12230431,2020-05-18,2020-05-20,1.0,1.001645


In [10]:
# Build expected (assigned_chartdate, subject_id) pairs
expected = date_df[['inpatient_ids']].reset_index().rename(columns={'date':'assigned_chartdate'})
expected = expected.explode('inpatient_ids').rename(columns={'inpatient_ids':'subject_id'})
expected = expected.dropna(subset=['subject_id']).copy()

# Normalize date column
expected['assigned_chartdate'] = pd.to_datetime(expected['assigned_chartdate']).dt.normalize()

# Prepare existing df_los (assigned_chartdate, subject_id)
existing = df_los.copy()
if existing.index.name is not None and 'assigned_chartdate' not in existing.columns:
    # If assigned_chartdate is the index, bring it back as a column
    existing = existing.reset_index()

if 'assigned_chartdate' not in existing.columns:
    raise KeyError('df_los does not contain a column or index named "assigned_chartdate". Cannot match. Please check df_los structure.')

existing['assigned_chartdate'] = pd.to_datetime(existing['assigned_chartdate']).dt.normalize()

# Coerce subject_id types to match for accurate comparison (fallback to string if mismatched)
try:
    expected['subject_id'] = expected['subject_id'].astype(existing['subject_id'].dtype)
except Exception:
    expected['subject_id'] = expected['subject_id'].astype(str)
    existing['subject_id'] = existing['subject_id'].astype(str)

# Deduplicate to avoid double counting
expected_pairs = expected[['assigned_chartdate','subject_id']].drop_duplicates()
existing_pairs = existing[['assigned_chartdate','subject_id']].drop_duplicates()

# Left anti-join to find expected pairs that are missing (records to add)
merged = expected_pairs.merge(existing_pairs, on=['assigned_chartdate','subject_id'], how='left', indicator=True)
missing = merged[merged['_merge'] == 'left_only'].drop(columns=['_merge'])

n_expected = len(expected_pairs)
n_existing_matches = len(expected_pairs) - len(missing)
n_missing = len(missing)

print(f'Expected date × inpatient pairs (unique): {n_expected}')
print(f'Existing matches (present in df_los): {n_existing_matches}')
print(f'Number of missing records to add: {n_missing}')

if n_missing > 0:
    print('\nMissing examples (first 20 rows):')
    display(missing.sort_values('assigned_chartdate').head(20))
else:
    print('\nNo missing records — df_los covers the inpatient lists for the given date range.')

# Optional: return missing DataFrame for use in creating completion records
missing = missing.reset_index(drop=True)


Expected date × inpatient pairs (unique): 79906
Existing matches (present in df_los): 43285
Number of missing records to add: 36621

Missing examples (first 20 rows):


,assigned_chartdate,subject_id
0,2020-05-19,10070587
39,2020-05-19,16799689
40,2020-05-19,17036580
41,2020-05-19,17057610
42,2020-05-19,17246547
43,2020-05-19,17289101
44,2020-05-19,17326039
45,2020-05-19,17434223
46,2020-05-19,17620255
47,2020-05-19,18092322


In [11]:
# Generate completion records (only fill assigned_chartdate and subject_id; other columns set to missing),
# and save as a new df_los CSV.
# Defensive: if 'missing' is not present, recompute the missing pairs.
if 'missing' not in globals():
    print("Variable 'missing' not found; recomputing missing pairs...")
    expected = date_df[['inpatient_ids']].reset_index().rename(columns={'date':'assigned_chartdate'})
    expected = expected.explode('inpatient_ids').rename(columns={'inpatient_ids':'subject_id'})
    expected = expected.dropna(subset=['subject_id']).copy()
    expected['assigned_chartdate'] = pd.to_datetime(expected['assigned_chartdate']).dt.normalize()
    existing = df_los.copy()
    if existing.index.name is not None and 'assigned_chartdate' not in existing.columns:
        existing = existing.reset_index()
    existing['assigned_chartdate'] = pd.to_datetime(existing['assigned_chartdate']).dt.normalize()
    expected_pairs = expected[['assigned_chartdate','subject_id']].drop_duplicates()
    existing_pairs = existing[['assigned_chartdate','subject_id']].drop_duplicates()
    merged = expected_pairs.merge(existing_pairs, on=['assigned_chartdate','subject_id'], how='left', indicator=True)
    missing = merged[merged['_merge'] == 'left_only'].drop(columns=['_merge']).reset_index(drop=True)

print('Preparing to append missing records to df_los (only assigned_chartdate and subject_id will be filled)...')
# Ensure df_los has an assigned_chartdate column
df_los = df_los.copy()
if df_los.index.name is not None and 'assigned_chartdate' not in df_los.columns:
    df_los = df_los.reset_index()

# Construct new_rows containing only assigned_chartdate and subject_id, and fill other df_los columns with NA
new_rows = missing.copy()
# Fill all other columns in new_rows with missing values to match df_los column structure
other_cols = [c for c in df_los.columns if c not in ['assigned_chartdate','subject_id']]
for c in other_cols:
    new_rows[c] = pd.NA

# Ensure column order matches df_los
new_rows = new_rows[df_los.columns.intersection(new_rows.columns).tolist() + [c for c in df_los.columns if c not in ['assigned_chartdate','subject_id'] and c not in new_rows.columns]] if len(new_rows)>0 else new_rows

# Concatenate and deduplicate (key = assigned_chartdate + subject_id)
combined = pd.concat([df_los, new_rows], ignore_index=True, sort=False)
before = df_los[['assigned_chartdate','subject_id']].drop_duplicates().shape[0]
after = combined[['assigned_chartdate','subject_id']].drop_duplicates().shape[0]
n_added = after - before
print(f'Number of missing pairs prepared to add: {len(new_rows)}')
print(f'Actually new rows (after dedup): {n_added}')

# Drop duplicates keeping original records first
combined = combined.drop_duplicates(subset=['assigned_chartdate','subject_id'], keep='first').copy()
# Sort by date and set assigned_chartdate as index for consistency
combined['assigned_chartdate'] = pd.to_datetime(combined['assigned_chartdate']).dt.normalize()
combined = combined.sort_values('assigned_chartdate')
combined = combined.set_index('assigned_chartdate')

# Save to CSV
out_csv = dest_dir / f"df_los_completed_{start_str}_to_{end_str}.csv"
combined.to_csv(out_csv)
print(f'Saved completed df_los to: {out_csv}')
print('Final df_los size (rows, cols):', combined.shape)

# Assign combined back to df_los for downstream cells to use (index is assigned_chartdate)
df_los = combined


Preparing to append missing records to df_los (only assigned_chartdate and subject_id will be filled)...
Number of missing pairs prepared to add: 36621
Actually new rows (after dedup): 36621


C:\Users\ZZY\AppData\Local\Temp\ipykernel_55200\423386084.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([df_los, new_rows], ignore_index=True, sort=False)


Saved completed df_los to: Predictions\df_los_completed_2020-05-19_to_2023-12-31.csv
Final df_los size (rows, cols): (79906, 5)


In [12]:
# Fill assigned_intime and assigned_outtime in df_los using Predictions icu patient episodes
from pathlib import Path
import pandas as pd

# path to the Predictions icu file (date-only episode list)
icu_src = Path("Predictions") / "icu_patients(re-assign)_dateonly.csv"
if not icu_src.exists():
    raise FileNotFoundError(f"Expected ICU episodes file not found: {icu_src}")

# load episodes and parse datetimes
icu = pd.read_csv(icu_src, low_memory=False)
for c in ['assigned_intime', 'assigned_outtime']:
    if c in icu.columns:
        icu[c] = pd.to_datetime(icu[c], errors='coerce').dt.normalize()

# avoid column name collisions when merging with df_los: rename episode columns to epi_*
rename_map = {}
if 'assigned_intime' in icu.columns:
    rename_map['assigned_intime'] = 'epi_intime'
if 'assigned_outtime' in icu.columns:
    rename_map['assigned_outtime'] = 'epi_outtime'
if rename_map:
    icu = icu.rename(columns=rename_map)

# create a simple episode id per subject (helps if a subject has multiple rows)
if 'subject_id' not in icu.columns:
    raise KeyError('icu file missing column subject_id')
icu = icu.sort_values(['subject_id', rename_map.get('assigned_intime', 'epi_intime') if rename_map else 'assigned_intime']).copy()
icu['episode_id'] = icu.groupby('subject_id').cumcount() + 1

# work on a copy of df_los and ensure assigned_chartdate is present as a column
try:
    df_check = df_los.copy()
except NameError:
    raise NameError('df_los not found in the notebook environment; run earlier cells to create it first')

if df_check.index.name == 'assigned_chartdate' or 'assigned_chartdate' not in df_check.columns:
    df_check = df_check.reset_index()
df_check['assigned_chartdate'] = pd.to_datetime(df_check['assigned_chartdate'], errors='coerce').dt.normalize()

# ensure subject_id types line up to avoid false mismatches
try:
    icu['subject_id'] = icu['subject_id'].astype(df_check['subject_id'].dtype)
except Exception:
    icu['subject_id'] = icu['subject_id'].astype(str)
    df_check['subject_id'] = df_check['subject_id'].astype(str)

# merge by subject_id (this may produce multiple candidate episodes per chartdate)
cols_to_merge = ['subject_id', 'episode_id']
if 'epi_intime' in icu.columns:
    cols_to_merge.append('epi_intime')
if 'epi_outtime' in icu.columns:
    cols_to_merge.append('epi_outtime')
merged = df_check.merge(icu[cols_to_merge], on='subject_id', how='left', sort=False)

# keep only rows where the chartdate falls within the episode interval: epi_intime <= chartdate <= epi_outtime (or epi_outtime is NA)
if 'epi_intime' in merged.columns:
    mask = (merged['epi_intime'].notna()) & (merged['epi_intime'] <= merged['assigned_chartdate']) & ((merged.get('epi_outtime').isna()) | (merged.get('epi_outtime') >= merged['assigned_chartdate']))
else:
    # no episode intime available, result will be empty
    mask = pd.Series([False] * len(merged), index=merged.index)
candidates = merged[mask].copy()

# if multiple candidate episodes exist for the same (assigned_chartdate, subject_id), pick the episode with the latest epi_intime (closest preceding admission)
if not candidates.empty:
    candidates = candidates.sort_values(['assigned_chartdate', 'subject_id'] + ([ 'epi_intime'] if 'epi_intime' in candidates.columns else []))
    best = candidates.drop_duplicates(subset=['assigned_chartdate', 'subject_id'], keep='last')
else:
    # empty DataFrame with expected columns
    best = candidates

# pivot best back to one row per (assigned_chartdate, subject_id) and assign the intime/outtime columns
if not best.empty:
    # select epi_* columns
    epi_cols = [c for c in ['epi_intime', 'epi_outtime'] if c in best.columns]
    best = best.set_index(['assigned_chartdate', 'subject_id'])[epi_cols]

# prepare result starting from df_check indexed by the two-key for easy assignment
res = df_check.set_index(['assigned_chartdate', 'subject_id']).copy()
# initialize columns if missing
for c in ['assigned_intime', 'assigned_outtime']:
    if c not in res.columns:
        res[c] = pd.NaT

# assign matched values where available
if not best.empty:
    # map epi columns back to assigned_* target columns
    mapping = {'assigned_intime': 'epi_intime', 'assigned_outtime': 'epi_outtime'}
    for target_col, epi_col in mapping.items():
        if epi_col in best.columns:
            res.loc[res.index.intersection(best.index), target_col] = best[epi_col].reindex(res.index).loc[res.index.intersection(best.index)]

# reset index back to normal form and assign back to df_los variable used by other cells
res = res.reset_index()

# count how many rows were filled
n_total = len(res)
n_filled_intime = res['assigned_intime'].notna().sum()
n_filled_outtime = res['assigned_outtime'].notna().sum()
print(f'Rows in df_los after processing: {n_total}; filled assigned_intime: {n_filled_intime}; filled assigned_outtime: {n_filled_outtime}')

# sort and set index for consistency with earlier workflow and save to the same completed CSV path used previously
res['assigned_chartdate'] = pd.to_datetime(res['assigned_chartdate']).dt.normalize()
res = res.sort_values('assigned_chartdate').set_index('assigned_chartdate')
out_csv = dest_dir / f"df_los_completed_{start_str}_to_{end_str}.csv"
res.to_csv(out_csv)
print(f'Updated df_los saved to: {out_csv}')

# assign back to df_los in the notebook so downstream cells can use it
df_los = res.copy()


Rows in df_los after processing: 79906; filled assigned_intime: 79906; filled assigned_outtime: 65466
Updated df_los saved to: Predictions\df_los_completed_2020-05-19_to_2023-12-31.csv


In [13]:
# Fill actual_los in df_los per rules:
# - For missing actual_los: actual_los = assigned_outtime - assigned_chartdate (days)
# - Clamp to [1,21]
# - If assigned_outtime is missing -> actual_los = 999
# - Preserve existing actual_los if present
# - Additionally: set predicted_los = 999 wherever actual_los == 999


if 'df_los' not in globals():
    raise

work = df_los.copy()
was_index = False
if work.index.name == 'assigned_chartdate' or 'assigned_chartdate' not in work.columns:
    work = work.reset_index()
    was_index = True

# normalize dates
work['assigned_chartdate'] = pd.to_datetime(work['assigned_chartdate'], errors='coerce').dt.normalize()
work['assigned_outtime'] = pd.to_datetime(work.get('assigned_outtime'), errors='coerce').dt.normalize()

# ensure actual_los numeric (NaN for non-numeric)
if 'actual_los' in work.columns:
    work['actual_los'] = pd.to_numeric(work['actual_los'], errors='coerce')
else:
    work['actual_los'] = pd.NA

# compute masks
mask_missing = work['actual_los'].isna()
before_missing = int(mask_missing.sum())

# Case 1: outtime missing -> set 999
mask_out_na = mask_missing & work['assigned_outtime'].isna()
work.loc[mask_out_na, 'actual_los'] = 999

# Case 2: outtime present -> compute difference and clamp
mask_compute = mask_missing & work['assigned_outtime'].notna()
if mask_compute.any():
    days = (work.loc[mask_compute, 'assigned_outtime'] - work.loc[mask_compute, 'assigned_chartdate']).dt.days
    # clamp to [1,21]
    days_clamped = days.clip(lower=1, upper=21)
    work.loc[mask_compute, 'actual_los'] = days_clamped.values

# finalize numeric type (keep float to preserve possible non-integer existing values)
work['actual_los'] = pd.to_numeric(work['actual_los'], errors='coerce')

after_missing = int(work['actual_los'].isna().sum())
filled = before_missing - after_missing
count_999 = int((work['actual_los'] == 999).sum())

# Set predicted_los = 999 where actual_los == 999
if 'predicted_los' in work.columns:
    work.loc[work['actual_los'] == 999, 'predicted_los'] = 999
else:
    work['predicted_los'] = pd.NA
    work.loc[work['actual_los'] == 999, 'predicted_los'] = 999

print(f"Missing before: {before_missing}, filled: {filled}, still missing: {after_missing}, count 999: {count_999}")

# restore index if needed and save
work['assigned_chartdate'] = pd.to_datetime(work['assigned_chartdate']).dt.normalize()
work = work.sort_values('assigned_chartdate')
if was_index:
    work = work.set_index('assigned_chartdate')

# assign back to df_los
df_los = work.copy()

# try saving to the same completed CSV path if available
try:
    out_csv = dest_dir / f"df_los_completed_{start_str}_to_{end_str}.csv"
    df_los.to_csv(out_csv)
    print(f"Updated df_los saved to: {out_csv}")
except Exception as e:
    print("Error:", e)


Missing before: 36621, filled: 36621, still missing: 0, count 999: 14440
Updated df_los saved to: Predictions\df_los_completed_2020-05-19_to_2023-12-31.csv


In [14]:
# NEW CELL: propagate predicted_los within each hospitalization episode using historical observed predictions
# For each episode (subject_id + assigned_intime + assigned_outtime), if at least one chartdate within the episode has a valid predicted_los (exclude sentinel 999), use the nearest such reference row to infer predicted_losfor other days by adding/subtracting the day offset: pred_target = pred_ref + (ref_date - target_date).days
# Only fill rows where predicted_los is missing (or sentinel) and actual_los != 999. Clamp to [1,21].
import pandas as pd
from pathlib import Path

# Defensive: require df_los from earlier cells
if 'df_los' not in globals():
    raise NameError('df_los not found in the notebook environment; run earlier cells to create it first')

work = df_los.copy()
was_index = False
if work.index.name == 'assigned_chartdate' or 'assigned_chartdate' not in work.columns:
    work = work.reset_index()
    was_index = True

# normalize date columns
work['assigned_chartdate'] = pd.to_datetime(work['assigned_chartdate'], errors='coerce').dt.normalize()
work['assigned_intime'] = pd.to_datetime(work.get('assigned_intime'), errors='coerce').dt.normalize()
work['assigned_outtime'] = pd.to_datetime(work.get('assigned_outtime'), errors='coerce').dt.normalize()

# Prepare predicted_los numeric; sentinel values (>=900) are treated as not usable for propagation
work['predicted_los'] = pd.to_numeric(work.get('predicted_los'), errors='coerce')

# Create a provenance column if missing
if 'pred_source' not in work.columns:
    work['pred_source'] = pd.NA
# mark original valid predictions (exclude sentinel 999 and similar)
orig_mask = work['predicted_los'].notna() & (work['predicted_los'] < 900)
work.loc[orig_mask & work['pred_source'].isna(), 'pred_source'] = 'original'

# Build episode key so rows belonging to the same hospitalization are grouped together
def fmt_date(d):
    return d.strftime('%Y-%m-%d') if pd.notna(d) else 'NA'

work['epi_intime_str'] = work['assigned_intime'].apply(fmt_date)
work['epi_outtime_str'] = work['assigned_outtime'].apply(fmt_date)
work['episode_key'] = work['subject_id'].astype(str) + '::' + work['epi_intime_str'] + '::' + work['epi_outtime_str']

# Eligible rows: predicted_los is missing or sentinel and actual_los != 999
missing_pred = work['predicted_los'].isna()
sentinel = work['predicted_los'] >= 900
eligible = (missing_pred | sentinel) & (work.get('actual_los') != 999)

filled = 0
# For each episode, find reference rows and propagate to eligible days using nearest reference
for epi, grp in work.groupby('episode_key'):
    grp = grp.sort_values('assigned_chartdate')
    refs = grp[grp['predicted_los'].notna() & (grp['predicted_los'] < 900)]
    if refs.empty:
        continue
    ref_dates = refs['assigned_chartdate'].tolist()
    ref_vals = refs['predicted_los'].tolist()
    for idx, row in grp.iterrows():
        if not eligible.loc[idx]:
            continue
        target = row['assigned_chartdate']
        # compute offsets (ref_date - target) in days
        diffs = [(rd - target).days for rd in ref_dates]
        abs_d = [abs(d) for d in diffs]
        pos = abs_d.index(min(abs_d))
        ref_pred = ref_vals[pos]
        offset = diffs[pos]
        candidate = ref_pred + offset
        try:
            cand_int = int(round(float(candidate)))
        except Exception:
            # if conversion fails, skip
            continue
        cand_int = max(1, min(21, cand_int))
        work.at[idx, 'predicted_los'] = cand_int
        work.at[idx, 'pred_source'] = 'propagated'
        filled += 1

remaining = int(work['predicted_los'].isna().sum())
print(f'Propagated predicted_los filled: {filled}; remaining missing: {remaining}')

# Save updated df_los to the completed path used earlier (catch PermissionError and fallback)
out_csv = Path('Predictions') / f"df_los_completed_{start_str}_to_{end_str}.csv"
work['assigned_chartdate'] = pd.to_datetime(work['assigned_chartdate']).dt.normalize()
work = work.sort_values('assigned_chartdate')
if was_index:
    work = work.set_index('assigned_chartdate')
df_los = work.copy()
try:
    df_los.to_csv(out_csv)
    print(f'Updated df_los saved to: {out_csv}')
except PermissionError as e:
    alt_out = Path('Predictions') / f"df_los_completed_{start_str}_to_{end_str}_propagated.csv"
    df_los.to_csv(alt_out)
    print(f'PermissionError when writing primary path; saved updated df_los to fallback path: {alt_out}')

Propagated predicted_los filled: 255; remaining missing: 21926
Updated df_los saved to: Predictions\df_los_completed_2020-05-19_to_2023-12-31.csv


In [15]:
# FILL REMAINING predicted_los using same-chartdate average (exclude sentinel 999)
import pandas as pd
from pathlib import Path
import numpy as np

# Ensure df_los is present
if 'df_los' not in globals():
    raise NameError('df_los not found in the notebook environment; run earlier cells to create it first')

work = df_los.copy()
was_index = False
if work.index.name == 'assigned_chartdate' or 'assigned_chartdate' not in work.columns:
    work = work.reset_index()
    was_index = True

# normalize date and numeric columns
work['assigned_chartdate'] = pd.to_datetime(work['assigned_chartdate'], errors='coerce').dt.normalize()
work['predicted_los'] = pd.to_numeric(work.get('predicted_los'), errors='coerce')
work['actual_los'] = pd.to_numeric(work.get('actual_los'), errors='coerce')

# sentinel values (>=900) are excluded from averaging and treated as missing for fill purpose
usable = work['predicted_los'].notna() & (work['predicted_los'] < 900)
# compute same-chartdate mean excluding sentinel
chart_means = work.loc[usable].groupby('assigned_chartdate')['predicted_los'].mean()

# target rows: predicted_los missing or sentinel, and actual_los != 999
is_missing_or_sentinel = work['predicted_los'].isna() | (work['predicted_los'] >= 900)
eligible = is_missing_or_sentinel & (work['actual_los'] != 999)

# map means to rows
work['chart_mean'] = work['assigned_chartdate'].map(chart_means)
to_fill_mask = eligible & work['chart_mean'].notna()
before_missing = int(work['predicted_los'].isna().sum()) + int((work['predicted_los'] >= 900).sum() if work['predicted_los'].dtype.kind in 'if' else 0)

# fill with rounded mean and clamp to [1,21]
if to_fill_mask.any():
    vals = work.loc[to_fill_mask, 'chart_mean'].astype(float).round().astype(int).clip(1,21)
    work.loc[to_fill_mask, 'predicted_los'] = vals.values
    work.loc[to_fill_mask & work['pred_source'].isna(), 'pred_source'] = 'chartdate_mean'

filled = int(to_fill_mask.sum())
remaining = int((work['predicted_los'].isna()).sum())
print(f'Chartdate-mean fill: filled={filled}; remaining missing={remaining}')

# cleanup temporary column
work = work.drop(columns=['chart_mean'], errors='ignore')

# save result (fallback if permission denied)
out_csv = Path('Predictions') / f"df_los_completed_{start_str}_to_{end_str}.csv"
if was_index:
    work = work.set_index('assigned_chartdate')
df_los = work.copy()
try:
    df_los.to_csv(out_csv)
    print(f'Updated df_los saved to: {out_csv}')
except PermissionError:
    alt = Path('Predictions') / f"df_los_completed_{start_str}_to_{end_str}_chartmean.csv"
    df_los.to_csv(alt)
    print(f'PermissionError writing primary path; saved to fallback: {alt}')

Chartdate-mean fill: filled=21926; remaining missing=0
Updated df_los saved to: Predictions\df_los_completed_2020-05-19_to_2023-12-31.csv


In [16]:
# save specified columns to CSV named los_test.csv
try:
    _df = df_los.copy()
except NameError:
    raise NameError("Variable df_los not found. Please run the cells that produce df_los first.")

# ensure assigned_chartdate is a column
if _df.index.name == 'assigned_chartdate' or 'assigned_chartdate' not in _df.columns:
    _df = _df.reset_index()

cols = ['assigned_chartdate', 'subject_id', 'predicted_los']
missing = [c for c in cols if c not in _df.columns]
if missing:
    raise KeyError(f"Missing columns in df_los: {missing}")

out_path = dest_dir / "los_test.csv" if 'dest_dir' in globals() else Path("los_test.csv")
_df.loc[:, cols].to_csv(out_path, index=False)
print(f"Saved {len(_df)} rows to: {out_path}")

Saved 79906 rows to: Predictions\los_test.csv
